# NB02b — EDA: Reviews

**Phase 2, Notebook 2 of 5.** Forensic exploration of the 105 774 Yelp reviews across Philadelphia and Tampa. Four figures saved to `outputs/figures/nb02b/`.

Context from NB02: review counts are thin for most restaurants (median 65 Philly, 41 Tampa), the corpus skews 5-star (44 k five-star vs 10 k one-star), and ~43% of restaurants are permanently closed — their reviews are historical signal only.

| Figure | Description |
|---|---|
| `fig5_review_length.png` | Review length by star bucket |
| `fig6_vader_sentiment.png` | VADER compound score by star rating |
| `fig7_tfidf_bigrams.png` | Top bigrams: 1-star vs 5-star |
| `fig8_volume_over_time.png` | Review volume over time by city |

Run cells top-to-bottom. Inspect each figure before proceeding.

## 1. Environment setup

Imports and VADER initialisation. VADER is a rule-based sentiment scorer — no model load, no GPU needed.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.auto import tqdm

ROOT      = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

PROCESSED = ROOT / "data" / "processed"
FIG_DIR   = ROOT / "outputs" / "figures" / "nb02b"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

sia = SentimentIntensityAnalyzer()
print("VADER ready.")
print(f"Figures dir : {FIG_DIR}")

## 2. Load reviews.parquet and inspect

Quick sanity check on the star distribution and date range before plotting. The 5-star skew (~42% of all reviews) is a known Yelp bias — people who had great experiences are more motivated to leave a review than those who had an average one.

In [ ]:
rev = pd.read_parquet(PROCESSED / "reviews.parquet")
print(f"Shape  : {rev.shape}")
print(f"Columns: {rev.columns.tolist()}")
print()

star_dist = rev["stars"].value_counts().sort_index()
print("Star distribution:")
for s, n in star_dist.items():
    pct = n / len(rev) * 100
    print(f"  {s:.0f}★  {n:>6}  ({pct:.1f}%)")

print()
print(f"Date range: {rev['date'].min().date()} → {rev['date'].max().date()}")
print(f"Cities    : {rev['city'].value_counts().to_dict()}")
print()
rev["char_len"] = rev["text"].str.len()
print(f"Review length  median={rev['char_len'].median():.0f}  "
      f"mean={rev['char_len'].mean():.0f}  max={rev['char_len'].max()}")

## 3. Figure 5 — Review length by star bucket

Box plot of character count by star rating. The question: do unhappy reviewers write more? A 1-star review that is detailed and specific is higher-quality training signal for the sentiment analysis than a one-liner. Yelp caps reviews at 5 000 characters — many 5-star reviews will hit that ceiling.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

star_order = [1.0, 2.0, 3.0, 4.0, 5.0]
data_by_star = [rev.loc[rev["stars"] == s, "char_len"].values for s in star_order]

bp = ax.boxplot(data_by_star, labels=[f"{int(s)}★" for s in star_order],
                patch_artist=True, showfliers=False, medianprops={"color": "black"})

palette = sns.color_palette("RdYlGn", n_colors=5)
for patch, color in zip(bp["boxes"], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel("Star rating")
ax.set_ylabel("Review length (characters)")
ax.set_title("Review length by star rating (outliers hidden)")
ax.axhline(5000, color="grey", linestyle="--", linewidth=0.8, label="Yelp 5 000-char cap")
ax.legend(fontsize=8)

out = FIG_DIR / "fig5_review_length.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
for s, d in zip(star_order, data_by_star):
    print(f"  {int(s)}★  median={np.median(d):.0f}  mean={np.mean(d):.0f}")

## 4. Figure 6 — VADER sentiment by star rating

Computes VADER compound score (−1 = maximally negative, +1 = maximally positive) for every review, then plots box plots by star bucket. If VADER aligns well with star ratings the boxes should be roughly monotonic from left to right — that validates VADER as a usable proxy for sentiment in this corpus without needing a fine-tuned model.

**Runs on CPU only, no GPU needed. Expect ~60–90 seconds for 105 k reviews.**

In [ ]:
# ── 4A: Compute VADER scores ─────────────────────────────────────────────────
print("Computing VADER scores...")
tqdm.pandas(desc="VADER")
rev["vader"] = rev["text"].fillna("").progress_apply(
    lambda t: sia.polarity_scores(t)["compound"]
)
print(f"Done. Score range: {rev['vader'].min():.3f} → {rev['vader'].max():.3f}")

In [ ]:
# ── 4B: Plot ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

data_by_star = [rev.loc[rev["stars"] == s, "vader"].values for s in star_order]
bp = ax.boxplot(data_by_star, labels=[f"{int(s)}★" for s in star_order],
                patch_artist=True, showfliers=False, medianprops={"color": "black"})

for patch, color in zip(bp["boxes"], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel("Star rating")
ax.set_ylabel("VADER compound score")
ax.set_title("VADER sentiment by star rating")
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8, label="Neutral")
ax.legend(fontsize=8)

out = FIG_DIR / "fig6_vader_sentiment.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
for s, d in zip(star_order, data_by_star):
    print(f"  {int(s)}★  median VADER={np.median(d):.3f}  "
          f"pct positive={np.mean(np.array(d) > 0)*100:.1f}%")

## 5. Figure 7 — Top TF-IDF bigrams: 1-star vs 5-star

Fits a TF-IDF vectoriser separately on 1-star and 5-star reviews and extracts the top 20 bigrams by mean TF-IDF score. Bigrams (two-word phrases) capture more specific language than single words — "never again" is more informative than "never" alone.

**Why TF-IDF rather than raw frequency?** Frequency rewards common words that appear everywhere. TF-IDF rewards words that are distinctive to this group of reviews relative to how commonly they appear in text generally — so "great food" appearing in 80% of 5-star reviews scores higher than "the" appearing in 100%.

In [ ]:
def top_bigrams(texts, n=20):
    vec = TfidfVectorizer(
        ngram_range=(2, 2),
        stop_words="english",
        max_features=10_000,
        min_df=5,
        strip_accents="unicode",
    )
    X = vec.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).flatten()
    terms = vec.get_feature_names_out()
    top_idx = scores.argsort()[::-1][:n]
    return [(terms[i], scores[i]) for i in top_idx]

one_star  = rev.loc[rev["stars"] == 1.0, "text"].fillna("").tolist()
five_star = rev.loc[rev["stars"] == 5.0, "text"].fillna("").tolist()
print(f"1-star reviews : {len(one_star)}")
print(f"5-star reviews : {len(five_star)}")

bg1 = top_bigrams(one_star)
bg5 = top_bigrams(five_star)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, data, title, color in zip(
    axes,
    [bg1, bg5],
    ["Top bigrams — 1-star reviews", "Top bigrams — 5-star reviews"],
    ["#d9534f", "#5cb85c"],
):
    terms, scores = zip(*data)
    ax.barh(range(len(terms)), scores, color=color, alpha=0.75, edgecolor="white")
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Mean TF-IDF score")
    ax.set_title(title)

fig.tight_layout()
out = FIG_DIR / "fig7_tfidf_bigrams.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")

## 6. Figure 8 — Review volume over time by city

Monthly review counts from 2005 to 2022, split by city. Two things to look for:

1. **Growth curve** — both cities should show rising review activity through ~2018–2019, reflecting Yelp's platform growth and the restaurants' increasing engagement.
2. **COVID dip** — a sharp drop in early 2020 followed by partial recovery. This is directly relevant to the scan: the corpus mixes pre-COVID, COVID, and post-COVID sentiment. A restaurant's 3-star average may include a period when it was operating at 50% capacity with stressed staff. The agent cannot distinguish this without a temporal filter — documented as a data-honesty finding.

In [ ]:
monthly = (
    rev.groupby([rev["date"].dt.to_period("M"), "city"])
    .size()
    .reset_index(name="count")
)
monthly["date"] = monthly["date"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(11, 4))
for city, grp in monthly.groupby("city"):
    ax.plot(grp["date"], grp["count"], label=city, linewidth=1.2)

# Mark COVID onset
ax.axvline(pd.Timestamp("2020-03-01"), color="red", linestyle="--",
           linewidth=1, label="COVID onset (Mar 2020)")

ax.set_xlabel("Date")
ax.set_ylabel("Reviews per month")
ax.set_title("Review volume over time by city")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()
fig.tight_layout()

out = FIG_DIR / "fig8_volume_over_time.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
# COVID impact
pre  = monthly[monthly["date"] < "2020-03-01"].groupby("city")["count"].mean()
post = monthly[monthly["date"] >= "2020-03-01"].groupby("city")["count"].mean()
print("Mean monthly reviews pre- vs post-COVID onset:")
for city in pre.index:
    print(f"  {city}: {pre[city]:.0f} → {post[city]:.0f}  "
          f"({(post[city]-pre[city])/pre[city]*100:+.0f}%)")

## 7. Summary card — NB02b

Concise findings for the horizon scan's data section, including data-honesty limitations surfaced by the EDA.

In [ ]:
print("=" * 58)
print("NB02b SUMMARY — Review corpus")
print("=" * 58)

for city, grp in rev.groupby("city"):
    print(f"\n── {city} ({len(grp)} reviews) ──")
    print(f"  Stars  mean={grp['stars'].mean():.2f}  "
          f"5-star share={( grp['stars']==5).mean()*100:.1f}%")
    print(f"  Length median={grp['char_len'].median():.0f} chars")
    if "vader" in grp.columns:
        print(f"  VADER  median={grp['vader'].median():.3f}  "
              f"pct positive={(grp['vader']>0).mean()*100:.1f}%")

print()
print("── Data honesty ──")
five_pct = (rev["stars"] == 5).mean() * 100
one_pct  = (rev["stars"] == 1).mean() * 100
print(f"  Star skew: {five_pct:.1f}% five-star vs {one_pct:.1f}% one-star.")
print("  Yelp exhibits positivity bias — satisfied customers review more often.")
print("  Corpus mixes pre-COVID, COVID, and post-COVID periods without labelling.")
print("  Reviews from closed restaurants are included — may reflect outdated quality.")
print("  Review counts are thin for most restaurants (median 65 Philly, 41 Tampa).")
print("  TF-IDF analysis is dominated by the high-review-count minority.")
print()
print("── Figures saved ──")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  {f.name}")
print()
print("Next: NB02c_eda_menus.ipynb")